## Dependencies

In [ ]:
from openpyxl import load_workbook
import pandas as pd
import re
from pyspark.sql import functions as F
from datetime import datetime, timezone
from delta.tables import DeltaTable

## Parametros

In [ ]:

source_path = "/Volumes/main/landing/clientes.xlsx"
sheet_name = "Clientes"

target_table = "catalog.dimensions.dim_clientes"

primary_key = "id_cliente"

# Solo se utilizan en SCD2
version_columns = ["ciudad", "categoria"]

load_strategy = "SCD2"  # SCD1 / SCD2 / REPLACE

process_operator = "Databricks_Job"

## File Read 
#### Esta sección utiliza openpyxl en modo read_only para evitar cargar innecesariamente el libro completo en memoria. También contempla filas vacías y encabezados.

In [ ]:
workbook = load_workbook(
    source_path,
    read_only=True,
    data_only=True
)

worksheet = workbook[sheet_name]

rows = worksheet.iter_rows(values_only=True)

headers = next(rows)

# Eliminar espacios de los encabezados
headers = [str(col).strip() if col is not None else "" for col in headers]

data = [
    row for row in rows
    if any(value is not None for value in row)
]

workbook.close()

pdf = pd.DataFrame(data, columns=headers)

if pdf.empty:
    raise ValueError("El archivo Excel no contiene registros.")

print(f"Registros leídos: {len(pdf)}")
print(f"Columnas detectadas: {len(pdf.columns)}")

## Normalización e inferencia de tipos
#### Normalizamos los nombres y dejamos que Spark infiera los tipos a partir de los valores leídos. Para columnas completamente vacías, asignamos temporalmente valores vacíos para que Spark pueda inferirlas como texto y luego los convertimos a NULL.

In [ ]:
def normalize_column(name):
    name = name.strip().lower()
    name = re.sub(r"[^a-zA-Z0-9]+", "_", name)
    return name.strip("_")


pdf.columns = [normalize_column(col) for col in pdf.columns]

if len(pdf.columns) != len(set(pdf.columns)):
    raise ValueError("Hay nombres de columnas duplicados después de normalizar.")

# Columnas completamente vacías
empty_columns = [
    col for col in pdf.columns
    if pdf[col].isna().all()
]

for col in empty_columns:
    pdf[col] = ""

df = spark.createDataFrame(pdf)

# Restaurar como NULL las columnas que estaban vacías
for col in empty_columns:
    df = df.withColumn(
        col,
        F.lit(None).cast("string")
    )

# Validaciones básicas
required_columns = [primary_key] + (
    version_columns if load_strategy == "SCD2" else []
)

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Faltan columnas requeridas: {missing_columns}")

if df.filter(F.col(primary_key).isNull()).limit(1).count() > 0:
    raise ValueError("La clave primaria contiene valores NULL.")

if df.groupBy(primary_key).count().filter("count > 1").limit(1).count() > 0:
    raise ValueError("El Excel contiene claves primarias duplicadas.")

print("Esquema detectado:")
df.printSchema()

## Detección de cambios y persistencia
### En esta sección se resuelve la estrategia elegida. La auditoría utiliza las mismas columnas para todas las estrategias, y agrega las columnas de vigencia cuando corresponde.

In [ ]:
process_ts = datetime.now(timezone.utc)

audit_columns = [
    "_inserted_at",
    "_updated_at",
    "_process_date",
    "_process_operator"
]

df = (
    df
    .withColumn("_inserted_at", F.lit(process_ts))
    .withColumn("_updated_at", F.lit(process_ts))
    .withColumn("_process_date", F.lit(process_ts))
    .withColumn("_process_operator", F.lit(process_operator))
)

table_exists = spark.catalog.tableExists(target_table)

business_columns = [
    col for col in df.columns
    if col not in audit_columns
]

# ============================================================
# REPLACE
# ============================================================

if load_strategy == "REPLACE":

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

# ============================================================
# SCD1
# ============================================================

elif load_strategy == "SCD1":

    if not table_exists:

        df.write.format("delta").mode("overwrite").saveAsTable(target_table)

    else:

        update_set = {
            col: f"source.`{col}`"
            for col in business_columns
            if col != primary_key
        }

        update_set.update({
            "_updated_at": "source._updated_at",
            "_process_date": "source._process_date",
            "_process_operator": "source._process_operator"
        })

        insert_values = {
            col: f"source.`{col}`"
            for col in df.columns
        }

        (
            DeltaTable.forName(spark, target_table)
            .alias("target")
            .merge(
                df.alias("source"),
                f"target.`{primary_key}` = source.`{primary_key}`"
            )
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsert(values=insert_values)
            .execute()
        )

# ============================================================
# SCD2
# ============================================================

elif load_strategy == "SCD2":

    hash_column = "_hash"

    df = df.withColumn(
        hash_column,
        F.sha2(
            F.to_json(F.struct(*version_columns)),
            256
        )
    )

    valid_to_default = F.to_timestamp(
        F.lit("9999-12-31 23:59:59")
    )

    if not table_exists:

        df = (
            df
            .withColumn("_valid_from", F.lit(process_ts))
            .withColumn("_valid_to", valid_to_default)
            .withColumn("_is_current", F.lit(True))
        )

        df.write.format("delta").mode("overwrite").saveAsTable(target_table)

    else:

        target = DeltaTable.forName(spark, target_table)

        # Versiones actuales
        current = (
            spark.table(target_table)
            .filter(F.col("_is_current") == True)
            .select(
                primary_key,
                F.col(hash_column).alias("_existing_hash")
            )
        )

        comparison = df.join(current, primary_key, "left")

        # Nuevos registros o registros con cambios versionables
        new_versions = (
            comparison
            .filter(
                F.col("_existing_hash").isNull() |
                (F.col(hash_column) != F.col("_existing_hash"))
            )
            .select(df.columns)
        )

        # Registros existentes cuyo contenido versionable cambió
        changed_keys = (
            comparison
            .filter(
                F.col("_existing_hash").isNotNull() &
                (F.col(hash_column) != F.col("_existing_hash"))
            )
            .select(primary_key)
        )

        # Registros sin cambios versionables
        unchanged = (
            comparison
            .filter(F.col(hash_column) == F.col("_existing_hash"))
            .select(df.columns)
        )

        # 1. Cerrar versiones anteriores
        (
            target.alias("target")
            .merge(
                changed_keys.alias("source"),
                f"""
                target.`{primary_key}` = source.`{primary_key}`
                AND target._is_current = true
                """
            )
            .whenMatchedUpdate(set={
                "_valid_to": "current_timestamp()",
                "_is_current": "false"
            })
            .execute()
        )

        # 2. Actualizar atributos que no generan versionado
        update_set = {
            col: f"source.`{col}`"
            for col in business_columns
            if col != primary_key
        }

        update_set.update({
            "_updated_at": "source._updated_at",
            "_process_date": "source._process_date",
            "_process_operator": "source._process_operator"
        })

        (
            target.alias("target")
            .merge(
                unchanged.alias("source"),
                f"""
                target.`{primary_key}` = source.`{primary_key}`
                AND target._is_current = true
                """
            )
            .whenMatchedUpdate(set=update_set)
            .execute()
        )

        # 3. Insertar las nuevas versiones
        new_versions = (
            new_versions
            .withColumn("_valid_from", F.lit(process_ts))
            .withColumn("_valid_to", valid_to_default)
            .withColumn("_is_current", F.lit(True))
        )

        new_versions.write.format("delta").mode("append").saveAsTable(target_table)